# Experimentación Computacional en *Machine Learning*
## Inspección y Particionamiento de Datos (`Breast Cancer`)

### En esta primera sección implementaremos:
### 1. **Auditoría inicial del dataset:** número de observaciones, variables, valores faltantes, duplicados y distribución de clases.
### 2. **Separación en tres subconjuntos:** Entrenamiento ($D_{train}$), Validación ($D_{validation}$) y Prueba ($D_{test}$).
### 3. **Estratificación: partición estratificada ($P(Y=c|D_{train}) \approx P(Y=c|D_{test})$) fijando una semilla (`random_state`).

# 1. Cargar el dataset `Breast Cancer`

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from ucimlrepo import fetch_ucirepo
import pandas as pd
from ucimlrepo import fetch_ucirepo

brest_cancer = fetch_ucirepo(id=17)
X = brest_cancer.data.features
y = brest_cancer.data.targets
serie_y = y["Diagnosis"]


# Calcular distribución de la variable objetivo ('Diagnosis')

conteo = serie_y.value_counts()
proporcion = serie_y.value_counts(normalize=True) * 100

distribucion = pd.DataFrame(
    {
        "Calidad (Clase)": conteo.index,
        "Frecuencia (n)": conteo.values,
        "Porcentaje (%)": proporcion.values.round(2),
    }
)

print(f"Total de muestras: {len(y)}")
print("\nDistribución de clases (Diagnosis):")
print(distribucion)

## 2. Separación en Train (70%), Validation (10%) y Test (20%)

### Como `train_test_split` divide los datos en dos bloques a la vez, aplicamos un **corte en dos etapas**:
### 1. **Primer corte:** Separamos el **80%** para entrenamiento (`X_train`, `y_train`) y dejamos un **20%** test (`X_test`, `y_test`).
### 2. **Segundo corte:** Dividimos ese 80% de train nuevamente (`test_size=0.1/0.8`) para obtener **10% Validación** y **70% para train**. Esto se hace en caso de requerir un conjunto de validación en el proceso.

> ### **Regla experimental:** El conjunto `X_test, y_test` queda aislado y no debe tocarse durante el ajuste de hiperparámetros ni la selección de modelos.

In [11]:

SEED = 42  # Semilla fija para garantizar reproducibilidad


# ==============================================================================
# ESTRATIFICADA (conserva proporciones de clase)
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)

# Proporcion_VALIDATION = porcentaje deseado del total de datos / porcentaje disponible en train
proporcion_VALIDATION = 0.1 / 0.8
X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=proporcion_VALIDATION,
    random_state=SEED,
    stratify=y_train,
)


In [ ]:
# ==============================================================================
# DISTRIBUCIÓN POR CLASE EN CADA PARTICIÓN (CONTEO Y PORCENTAJE)
CLASE = "Diagnosis"
conteos = pd.DataFrame(
    {
        "Original (n)": y[CLASE].value_counts(),
        "Train 70% (n)": y_train[CLASE].value_counts(),
        "Val 10% (n)": y_val[CLASE].value_counts(),
        "Test 20% (n)": y_test[CLASE].value_counts(),
    }
).sort_index()

porcentajes = (
    pd.DataFrame(
        {
            "Original (%)": y[CLASE].value_counts(normalize=True) * 100,
            "Train 70% (%)": y_train[CLASE].value_counts(normalize=True) * 100,
            "Val 10% (%)": y_val[CLASE].value_counts(normalize=True) * 100,
            "Test 20% (%)": y_test[CLASE].value_counts(normalize=True) * 100,
        }
    )
    .sort_index()
    .round(2)
)

print("=" * 65)
print("CONTEO DE MUESTRAS POR CLASE")
print("=" * 65)
print(conteos)

print("\n" + "=" * 65)
print("PROPORCIÓN POR CLASE (%)")
print("=" * 65)
print(porcentajes)

## 3. Primer Modelo Base: Support Vector Machine (SVM) — Train vs. Test
[Documentación oficial de Support Vector Machines (SVM) en Scikit-learn](https://scikit-learn.org/stable/modules/svm.html)

### En esta sección entrenaremos un clasificador **SVM con kernel RBF** (`SVC`) siguiendo el flujo experimental paso a paso  para observar:
### 1. SVM calcula distancias entre puntos, en este modelo no se hacen preprocesamiento.
### 2. **Desempeño en Entrenamiento vs. Prueba:** Evaluaremos el modelo tanto en `Train` como en `Test` para verificar si existe sobreajuste (*overfitting*) y cómo afecta el desbalanceo de clases a las métricas (**Accuracy** vs. **F1-macro**).

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# Extraer vectores 1D de las etiquetas (ucimlrepo devuelve DataFrames)
y_tr = y_train[CLASE].values
y_te = y_test[CLASE].values

# ==============================================================================
# PASO 1: ENTRENAMIENTO DEL CLASIFICADOR SVM
# ==============================================================================
# ==============================================================================
# DEFINICIÓN Y ENTRENAMIENTO DEL CLASIFICADOR SVM (SUPPORT VECTOR MACHINE)
# ==============================================================================

# 1. Instanciar el modelo con sus hiperparámetros experimentales:
svm_model = SVC(
    kernel="rbf",  # Kernel de Base Radial (Gaussiano): permite trazar fronteras
                   # de decisión no lineales proyectando los datos a mayor dimensión.
    C=1.0,         # Parámetro de regularización (penalización por errores):
                   #   - C bajo  -> margen más ancho, tolera más errores (evita overfitting).
                   #   - C alto  -> margen más estricto, intenta clasificar todo bien (riesgo de overfitting).
    gamma="scale",  # Alcance de influencia de cada muestra de entrenamiento:
                    #   - 'scale' calcula automáticamente 1 / (n_features * X.var()).
                    #   - Valores altos crean fronteras muy ajustadas a cada punto.
    random_state=SEED,  # Semilla aleatoria para garantizar reproducibilidad interna.
)

# 2. Entrenar (ajustar) el modelo usando ÚNICAMENTE los datos de entrenamiento:
# Aquí el algoritmo encuentra los "vectores de soporte" (los puntos cercanos a la frontera)
# que definen los hiperplanos de separación entre las clases.
# Nota: recuerda pasar X_train_scaled si estás usando variables estandarizadas.
svm_model.fit(X_train, y_tr)

# ==============================================================================
# PASO 2: PREDICCIONES EN AMBOS CONJUNTOS (TRAIN Y TEST)
# ==============================================================================
y_pred_train = svm_model.predict(X_train)
y_pred_test = svm_model.predict(X_test)

# ==============================================================================
# PASO 3: COMPARACIÓN DE RENDIMIENTO (TRAIN VS. TEST)
# ==============================================================================
resultados_svm = pd.DataFrame(
    {
        "Métrica": [
            "Accuracy",
            "Precision (macro)",
            "Recall (macro)",
            "F1-score (macro)",
        ],
        "Train (70%)": [
            accuracy_score(y_tr, y_pred_train),
            precision_score(y_tr, y_pred_train, average="macro", zero_division=0),
            recall_score(y_tr, y_pred_train, average="macro", zero_division=0),
            f1_score(y_tr, y_pred_train, average="macro", zero_division=0),
        ],
        "Test (20%)": [
            accuracy_score(y_te, y_pred_test),
            precision_score(y_te, y_pred_test, average="macro", zero_division=0),
            recall_score(y_te, y_pred_test, average="macro", zero_division=0),
            f1_score(y_te, y_pred_test, average="macro", zero_division=0),
        ],
    }
)

resultados_svm["Diferencia (Train - Test)"] = (
    resultados_svm["Train (70%)"] - resultados_svm["Test (20%)"]
)

print("=" * 75)
print("RENDIMIENTO DE SVM (RBF, C=1.0) — ENTRENAMIENTO VS. PRUEBA")
print("=" * 75)
print(resultados_svm.round(4).to_string(index=False))

# 4. Preprocesamiento de los datos con Estandarización de variables: `StandardScaler()`

### La instrucción `scaler = StandardScaler()` crea un transformador que **estandariza** cada variable numérica (puntuación *Z-score*) para que tenga **media $\mu = 0$** y **desviación estándar $\sigma = 1$**, aplicando la fórmula:

$$
\large
z = \frac{x - \mu}{\sigma}
$$

#### ¿Por qué es indispensable en este experimento?
### 1. **Evita sesgos por escala:** Algoritmos basados en distancias o productos internos (como **SVM** y **KNN**) son sensibles a la magnitud de las variables. 
   * ### Sin estandarizar, una característica con valores grandes (ej. `0` a `1000`) dominaría el cálculo sobre otra con valores pequeños (ej. `0` a `1`).
### 2. **Previene la fuga de datos (*Data Leakage*):** Al guardar el escalador en el objeto `scaler`, garantizamos que los parámetros $\mu$ y $\sigma$ se calculen **exclusivamente** sobre el conjunto de entrenamiento (`fit_transform`) y se apliquen sin recalcular al conjunto de validación y prueba (`transform`):
   * ### `X_train_scaled = scaler.fit_transform(X_train)` $\rightarrow$ aprende $\mu_{train}, \sigma_{train}$ y transforma.
   * ### `X_test_scaled = scaler.transform(X_test)` $\rightarrow$ aplica $\mu_{train}, \sigma_{train}$ sin ver los datos de prueba.


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

# Extraer vectores 1D de las etiquetas (ucimlrepo devuelve DataFrames)
y_tr = y_train[CLASE].values
y_te = y_test[CLASE].values


# ==============================================================================
# PASO 1: ESTANDARIZACIÓN
# ==============================================================================

# ==============================================================================
# ESTANDARIZACIÓN DE CARACTERÍSTICAS (SIN FUGA DE DATOS / DATA LEAKAGE)
# ==============================================================================
# Muchos algoritmos (como SVM, KNN o Redes Neuronales) calculan distancias entre
# muestras. Si una variable está en miles y otra entre 0 y 1, la variable grande
# dominará el cálculo. StandardScaler transforma cada columna a media = 0 y
# desviación estándar = 1 usando: z = (x - media) / desviacion_estandar.

# 1. Crear el objeto escalador
scaler = StandardScaler()

# 2. FIT + TRANSFORM sobre el conjunto de ENTRENAMIENTO (X_train):
#    - .fit()       -> Calcula y guarda la media (μ) y desviación estándar (σ)
#                      ÚNICAMENTE de los datos de entrenamiento.
#    - .transform() -> Aplica la fórmula z = (x - μ) / σ a X_train.
X_train_scaled = scaler.fit_transform(X_train)


# 3. SOLO TRANSFORM sobre el conjunto de PRUEBA (X_test):
#    - REGLA DE ORO: NUNCA usar .fit() ni .fit_transform() en X_test (ni en X_val).
#    - Usamos exactamente la μ y σ aprendidas en X_train para simular que X_test
#      son datos nuevos del mundo real y evitar filtrar información del futuro
#      hacia el entrenamiento (Data Leakage).
X_test_scaled = scaler.transform(X_test)


# ==============================================================================
# PASO 2: ENTRENAMIENTO DEL CLASIFICADOR SVM
# ==============================================================================
svm_model = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=SEED)
svm_model.fit(X_train_scaled, y_tr)

# ==============================================================================
# PASO 3: PREDICCIONES EN AMBOS CONJUNTOS (TRAIN Y TEST)
# ==============================================================================
y_pred_train = svm_model.predict(X_train_scaled)
y_pred_test = svm_model.predict(X_test_scaled)

# ==============================================================================
# PASO 4: COMPARACIÓN DE RENDIMIENTO (TRAIN VS. TEST)
# ==============================================================================
resultados_svm_ss = pd.DataFrame(
    {
        "Métrica": [
            "Accuracy",
            "Precision (macro)",
            "Recall (macro)",
            "F1-score (macro)",
        ],
        "Train (70%)": [
            accuracy_score(y_tr, y_pred_train),
            precision_score(y_tr, y_pred_train, average="macro", zero_division=0),
            recall_score(y_tr, y_pred_train, average="macro", zero_division=0),
            f1_score(y_tr, y_pred_train, average="macro", zero_division=0),
        ],
        "Test (20%)": [
            accuracy_score(y_te, y_pred_test),
            precision_score(y_te, y_pred_test, average="macro", zero_division=0),
            recall_score(y_te, y_pred_test, average="macro", zero_division=0),
            f1_score(y_te, y_pred_test, average="macro", zero_division=0),
        ],
    }
)

resultados_svm_ss["Diferencia (Train - Test)"] = (
    resultados_svm_ss["Train (70%)"] - resultados_svm_ss["Test (20%)"]
)

print("=" * 75)
print("RENDIMIENTO DE SVM (RBF, C=1.0) — ENTRENAMIENTO VS. PRUEBA")
print("=" * 75)
print(resultados_svm_ss.round(4).to_string(index=False))

In [ ]:
print(resultados_svm.round(4).to_string(index=False))
print(resultados_svm_ss.round(4).to_string(index=False))

### Catálogo de escaladores y transformaciones

[Documentación oficial de `sklearn.preprocessing` en Scikit-learn](https://scikit-learn.org/stable/api/sklearn.preprocessing.html)


| Técnica en `sklearn.preprocessing` | Fórmula / Mecanismo | Cuándo usarlo / Qué efecto tiene |
| :--- | :--- | :--- |
| **`StandardScaler`** (Línea base) | $z = \dfrac{x - \mu}{\sigma}$ | Centra en media $0$ y desviación estándar $1$. Sensible a valores atípicos (*outliers*). |
| **`MinMaxScaler`** | $x' = \dfrac{x - x_{\min}}{x_{\max} - x_{\min}}$ | Comprime todas las variables al rango exacto $[0, 1]$. Muy sensible a extremos aislados. |
| **`RobustScaler`** | $x' = \dfrac{x - Q_2}{Q_3 - Q_1}$ | Centra usando la **mediana** ($Q_2$) y escala con el **rango intercuartílico** ($IQR$). Ideal cuando hay *outliers*. |
| **`MaxAbsScaler`** | $x' = \dfrac{x}{\lvert x_{\max} \rvert}$ | Escala al rango $[-1, 1]$ sin desplazar el centro (conserva ceros / dispersión). |
| **`PowerTransformer` (Yeo-Johnson)** | Transformación no lineal paramétrica | Corrige asimetría (*skewness*) para que las variables se parezcan a una distribución normal antes de estandarizar. |
| **`QuantileTransformer` (Normal)** | Mapeo no lineal por cuantiles | Fuerza cada variable a seguir una distribución Gaussiana; neutraliza por completo *outliers* extremos. |

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler_mm = MinMaxScaler(feature_range=(0, 1))

In [ ]:
from sklearn.preprocessing import PowerTransformer

# method='yeo-johnson' acepta valores positivos, cero y negativos (a diferencia de 'box-cox')
# standardize=True (por defecto) deja las variables con media 0 y desv. est. 1
scaler_pwr = PowerTransformer(method="yeo-johnson", standardize=True)


# Ejercicio 1. 
## Aplicar las estandarizaciones de datos MinMaxScaler y PowerTransformer y comparar los resultados de entrenar 2 modelos de SVM con la entrada de estos datos.
## Comparar el rendimiento de estos 2 modelos con los modelos anteriores: sin StandarScaler y con StandardScaler

In [16]:
# TODO: 







# Ejercicio 2. Comparación de Preprocesamientos con un Clasificador KNN

## [Guía oficial de Nearest Neighbors (KNN) en Scikit-learn](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsClassifier.html#sklearn.neighbors.KNeighborsClassifier)

### En el ejercicio anterior observamos cómo el escalado de variables impacta directamente en el desempeño de un modelo basado en distancias (**SVM**). Ahora evaluaremos un segundo algoritmo sensible a distancias: **K-Nearest Neighbors (`KNeighborsClassifier`)**.

### Instrucciones:
### 1. **Preprocesamiento sin fuga de datos (*Data Leakage*):**
   * ### Sin escalar (línea base)
   * ### Crea un escalador **`StandardScaler()`** y aplícalo ajustando **únicamente** con `X_train` (`.fit_transform`) y transformando `X_test` (`.transform`).
   * ### Crea un escalador **`MinMaxScaler()`** y aplícalo ajustando **únicamente** con `X_train` (`.fit_transform`) y transformando `X_test` (`.transform`).
   * ### Crea un transformador **`PowerTransformer()`** y aplícalo siguiendo la misma regla estricta (`X_train` vs. `X_test`).
### 2. **Entrenamiento de 4 modelos KNN:**
   * ### Entrenar los clasificadores  `KNeighborsClassifier(n_neighbors=5)` utilizando los datos transformados con los datos del paso 1.

### 3. **Evaluación y análisis comparativo:**
   * ### Calcula las métricas en `Train` y `Test` (Accuracy, Precisón, Recall, F1-macro ) para todos los modelos **KNN**.
   * ### Construye una tabla comparativa que integre los **experimentos**:
        ### 1.  `SVM` sin escalar (línea base)
        ### 2.  `SVM` + `StandardScaler`
        ### 3.  `SVM` + `MinMaxScaler`
        ### 4.  `SVM` + `PowerTransformer`
        ### 5.  `KNN (k=5)` + sin escalar (línea base)
        ### 6.  `KNN (k=5)` + `StandardScaler`
        ### 7.  `KNN (k=5)` + `MinMaxScaler`
        ### 8.  `KNN (k=5)` + `PowerTransformer`
   * ### **Pregunta de análisis:** ¿Cuál combinación de *preprocesamiento + algoritmo* logra mejor generalización en `Test` y qué diferencia observas entre la brecha *Train - Test* de `KNN` frente a `SVM`?

In [36]:
# TODO: 

from sklearn.preprocessing import MinMaxScaler, PowerTransformer
from sklearn.neighbors import KNeighborsClassifier
# Ejemplo de uso del clasificador KNN
knn_mm = KNeighborsClassifier(n_neighbors=5)
# Los métodos fit  y predict son similares a los usados en el clasificador SVM